# Instructions

## Ivan's Message
In terms of analysis can you prioritize one check:

For individual parcelated model,
  Take 95th percentile per parcel as a scalar stat (instead of mean)
  Estimate global mean E across parcels
  Divide each parcel E by global E
  Demean the parcels per subject
  Run PCA in (subjects x parcels)

I want to know how many principle components explain most of our variance (~96-99 % or so), and what is a linear correlation with treatment outcomes of:
  Pure global mean E (unweighted by anything)
  Weighted global mean E
  And same for PC 1 to X (whatever is a meaningful X)

Later we could also check the spatial weights of PC of interest to identify localization.

For NOW, i am most interested in just knowning number of PC that explain most of the data and if correlation with global mean E is meaningful.

In the future we probably want to substitute linear correlation with something like:
Delta GCI ~ E_magn + charge + E_magn : charge
Optionally, “+ age * sex” or “+ (1|age*sex)”
(Per parcel or PC)


In [ ]:
"""Parcel-wise electric-field summaries, PCA, and outcome correlations.

The extraction functions load a SimNIBS ``Msh`` stored inside each gPC HDF5
file under ``mesh_roi``. Scalar fields such as ``magnE_mean`` are normally
element data in this representation. Parcel membership must therefore also be
defined over elements, either by an element label field, custom element tags,
or a caller-supplied mask function.

The PCA pipeline is:

1. Calculate the requested percentile of E in every parcel.
2. Calculate unweighted and parcel-size-weighted global means of those parcel
   percentiles.
3. Divide each parcel percentile by the subject's unweighted global mean.
4. Demean the normalized parcels within each subject.
5. Fit PCA to the resulting subjects x parcels matrix.

No rows, parcels, missing values, or non-finite values are silently removed.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy.stats import norm, pearsonr
from sklearn.decomposition import PCA


@dataclass(frozen=True)
class ParcelSummary:
    """Per-subject parcel statistics and their global summaries."""

    parcel_p95: pd.DataFrame
    parcel_size: pd.DataFrame
    global_metrics: pd.DataFrame


@dataclass(frozen=True)
class ParcelPCA:
    """PCA inputs and fitted outputs."""

    relative_parcels: pd.DataFrame
    subject_demeaned_parcels: pd.DataFrame
    scores: pd.DataFrame
    loadings: pd.DataFrame
    variance: pd.DataFrame
    model: PCA


def infer_subject_id(hdf5_file: str | Path) -> str:
    """Infer a modeled subject ID such as ``subj-cat-012-001`` from a path."""
    matches = re.findall(r"subj-[A-Za-z0-9]+-\d{3}-\d{3}", str(hdf5_file))
    unique_matches = sorted(set(matches))
    if len(unique_matches) != 1:
        raise ValueError(f"Expected exactly one subject ID in path, found {unique_matches}: {hdf5_file}")
    return unique_matches[0]


def inspect_hdf5_mesh(
    hdf5_file: str | Path,
    *,
    mesh_key: str = "mesh_roi",
    mesh_loader: Callable | None = None,
) -> dict[str, object]:
    """Load one HDF5 mesh and report fields, locations, tags, and dimensions."""
    mesh = load_hdf5_mesh(hdf5_file, mesh_key=mesh_key, mesh_loader=mesh_loader)
    element_field_names = {field.field_name for field in mesh.elmdata}
    node_field_names = {field.field_name for field in mesh.nodedata}
    fields = {}
    for name, field in mesh.field.items():
        values = np.asarray(field.value)
        location = "element" if name in element_field_names else "node" if name in node_field_names else "unknown"
        fields[name] = {"shape": values.shape, "class": type(field).__name__, "location": location}
    return {
        "hdf5_file": str(hdf5_file),
        "mesh_key": mesh_key,
        "n_nodes": int(mesh.nodes.nr),
        "n_elements": int(mesh.elm.nr),
        "element_types": np.unique(np.asarray(mesh.elm.elm_type)).astype(int).tolist(),
        "element_tags": np.unique(np.asarray(mesh.elm.tag1)).astype(int).tolist(),
        "fields": fields,
    }


def parcel_masks_from_element_field(
    mesh,
    field_name: str,
    *,
    label_names: Mapping[int, str] | None = None,
    exclude_labels: Sequence[int] = (0,),
) -> dict[str, np.ndarray]:
    """Build element masks from a scalar integer-valued ElementData field."""
    labels, location = _get_scalar_field(mesh, field_name, "parcel labels", require_nonnegative=True)
    if location != "element":
        raise ValueError(f"Parcel field {field_name!r} must be ElementData, got {location} data")
    return _parcel_masks_from_labels(labels, label_names=label_names, exclude_labels=exclude_labels)


def parcel_masks_from_element_tags(
    mesh,
    *,
    label_names: Mapping[int, str] | None = None,
    include_tags: Sequence[int] | None = None,
) -> dict[str, np.ndarray]:
    """Build parcel masks from ``mesh.elm.tag1``.

    Use this only when ``tag1`` contains parcel IDs. In a standard SimNIBS head
    mesh, these are tissue tags and must not be interpreted as brain parcels.
    """
    labels = np.asarray(mesh.elm.tag1).squeeze()
    exclude_labels = () if include_tags is None else tuple(set(np.unique(labels)).difference(include_tags))
    return _parcel_masks_from_labels(labels, label_names=label_names, exclude_labels=exclude_labels)


def build_parcel_summary(
    hdf5_files: Sequence[str | Path],
    *,
    parcel_mask_getter: Callable,
    subject_ids: Sequence[str] | None = None,
    field_name: str = "magnE_mean",
    percentile: float = 95.0,
    mesh_key: str = "mesh_roi",
    mesh_loader: Callable | None = None,
) -> ParcelSummary:
    """Extract one percentile value per parcel from each HDF5 ``mesh_roi``.

    Parameters
    ----------
    hdf5_files
        One gPC ``.hdf5`` path per modeled subject.
    parcel_mask_getter
        Callable receiving the loaded mesh and returning ``{parcel_name: mask}``.
        Each mask must align with the field values. Convenience functions are
        provided for an element label field and custom element tags.
    subject_ids
        IDs aligned with ``hdf5_files``. If omitted, IDs are inferred from the
        paths with :func:`infer_subject_id`.
    field_name
        Scalar E-field in each mesh, normally the ElementData field
        ``magnE_mean``.
    percentile
        Parcel-level percentile. The requested analysis uses 95.
    mesh_key
        HDF5 group containing the mesh. Default is ``mesh_roi``.
    mesh_loader
        Optional callable accepting ``(path, mesh_key)`` and returning a Msh.
        The default is ``simnibs.Msh.read_hdf5``.

    Returns
    -------
    ParcelSummary
        Parcel percentile and physical-size matrices plus two global summaries.
        ``global_mean_p95_unweighted`` gives each parcel equal influence;
        ``global_mean_p95_weighted`` weights each parcel P95 by its total
        element volume or surface area.
    """
    hdf5_files = [Path(path) for path in hdf5_files]
    if not hdf5_files:
        raise ValueError("hdf5_files is empty")
    if not callable(parcel_mask_getter):
        raise TypeError("parcel_mask_getter must be callable")
    if not 0.0 <= percentile <= 100.0:
        raise ValueError(f"percentile must be between 0 and 100, got {percentile}")

    if subject_ids is None:
        subject_ids = [infer_subject_id(path) for path in hdf5_files]
    else:
        subject_ids = [str(subject_id) for subject_id in subject_ids]
    if len(subject_ids) != len(hdf5_files):
        raise ValueError(f"subject_ids has {len(subject_ids)} entries but hdf5_files has {len(hdf5_files)}")
    duplicate_ids = pd.Index(subject_ids)[pd.Index(subject_ids).duplicated()].unique().tolist()
    if duplicate_ids:
        raise ValueError(f"Duplicate modeled subject IDs: {duplicate_ids}")

    p95_rows = []
    size_rows = []
    expected_parcels = None
    for subject_id, hdf5_file in zip(subject_ids, hdf5_files, strict=True):
        mesh = load_hdf5_mesh(hdf5_file, mesh_key=mesh_key, mesh_loader=mesh_loader)
        field, location = _get_scalar_field(mesh, field_name, subject_id, require_nonnegative=True)
        spatial_weights = _get_spatial_weights(mesh, location, subject_id)
        masks = _validate_parcel_masks(parcel_mask_getter(mesh), field.size, subject_id)
        parcel_p95, parcel_size = _summarize_one_subject(field, spatial_weights, masks, percentile, subject_id)

        parcel_names = tuple(parcel_p95.index)
        if expected_parcels is None:
            expected_parcels = parcel_names
        elif parcel_names != expected_parcels:
            raise ValueError(f"Parcel names or order differ for {subject_id}")
        p95_rows.append(parcel_p95.rename(subject_id))
        size_rows.append(parcel_size.rename(subject_id))

    parcel_p95 = pd.DataFrame(p95_rows)
    parcel_size = pd.DataFrame(size_rows)
    unweighted = parcel_p95.mean(axis=1)
    weighted = (parcel_p95 * parcel_size).sum(axis=1) / parcel_size.sum(axis=1)
    global_metrics = pd.DataFrame({
        "global_mean_p95_unweighted": unweighted,
        "global_mean_p95_weighted": weighted,
    })
    _require_finite_frame(global_metrics, "global metrics")
    if (global_metrics <= 0).any().any():
        bad = global_metrics.index[(global_metrics <= 0).any(axis=1)].tolist()
        raise ValueError(f"Global E must be positive before normalization; invalid subjects: {bad}")
    return ParcelSummary(parcel_p95, parcel_size, global_metrics)


def fit_parcel_pca(parcel_p95: pd.DataFrame) -> ParcelPCA:
    """Normalize parcel P95 values within subject and fit unscaled PCA."""
    _validate_subject_parcel_frame(parcel_p95)
    global_mean = parcel_p95.mean(axis=1)
    if (global_mean <= 0).any():
        raise ValueError(f"Every subject must have positive global E; invalid: {global_mean[global_mean <= 0].index.tolist()}")

    relative = parcel_p95.div(global_mean, axis="index")
    subject_demeaned = relative.sub(relative.mean(axis=1), axis="index")
    if not np.allclose(subject_demeaned.mean(axis=1), 0.0, atol=1e-12):
        raise RuntimeError("Subject demeaning failed")
    if np.isclose(subject_demeaned.var(axis=0, ddof=1).sum(), 0.0):
        raise ValueError("The normalized parcel profiles have no between-subject variance")

    model = PCA(svd_solver="full")
    score_values = model.fit_transform(subject_demeaned.to_numpy(dtype=float))
    pc_names = [f"PC{i}" for i in range(1, score_values.shape[1] + 1)]
    scores = pd.DataFrame(score_values, index=parcel_p95.index, columns=pc_names)
    loadings = pd.DataFrame(model.components_.T, index=parcel_p95.columns, columns=pc_names)
    variance = pd.DataFrame({
        "component": pc_names,
        "explained_variance": model.explained_variance_,
        "explained_variance_ratio": model.explained_variance_ratio_,
        "cumulative_variance_ratio": np.cumsum(model.explained_variance_ratio_),
    }).set_index("component")
    return ParcelPCA(relative, subject_demeaned, scores, loadings, variance, model)


def components_for_variance(pca_result: ParcelPCA, threshold: float) -> int:
    """Return the smallest number of PCs reaching a cumulative threshold."""
    if not 0.0 < threshold <= 1.0:
        raise ValueError(f"threshold must be in (0, 1], got {threshold}")
    cumulative = pca_result.variance["cumulative_variance_ratio"].to_numpy()
    if cumulative[-1] + 1e-12 < threshold:
        raise ValueError(f"PCA reaches only {cumulative[-1]:.6f}, below threshold {threshold:.6f}")
    return int(np.searchsorted(cumulative, threshold, side="left") + 1)


def correlate_predictors(
    predictors: pd.DataFrame,
    outcomes: pd.DataFrame,
    *,
    subject_col: str = "subj_id",
    outcome_col: str = "cgi_change",
) -> pd.DataFrame:
    """Run strict Pearson correlations and return r, p, and Fisher 95% CIs."""
    aligned_outcome = _align_outcomes(predictors.index, outcomes, subject_col, outcome_col)
    rows = []
    for predictor_name in predictors.columns:
        x = pd.to_numeric(predictors[predictor_name], errors="raise").to_numpy(dtype=float)
        y = aligned_outcome.to_numpy(dtype=float)
        if not np.isfinite(x).all():
            raise ValueError(f"Predictor {predictor_name!r} contains non-finite values")
        if np.ptp(x) == 0:
            raise ValueError(f"Predictor {predictor_name!r} has zero variance")
        result = pearsonr(x, y)
        ci_low, ci_high = _pearson_fisher_ci(float(result.statistic), len(x))
        rows.append({
            "predictor": predictor_name,
            "n": len(x),
            "r": float(result.statistic),
            "ci_95_low": ci_low,
            "ci_95_high": ci_high,
            "p_value": float(result.pvalue),
        })
    correlations = pd.DataFrame(rows).set_index("predictor")
    correlations["p_fdr_bh"] = _benjamini_hochberg(correlations["p_value"])
    return correlations


def global_and_pc_predictors(summary: ParcelSummary, pca_result: ParcelPCA, n_pcs: int = 0) -> pd.DataFrame:
    """Combine the two global metrics with PC1..PCn for correlation testing."""
    if not 0 <= n_pcs <= pca_result.scores.shape[1]:
        raise ValueError(f"n_pcs must be between 0 and {pca_result.scores.shape[1]}, got {n_pcs}")
    if not summary.global_metrics.index.equals(pca_result.scores.index):
        raise ValueError("Subject order differs between global metrics and PCA scores")
    return pd.concat([summary.global_metrics, pca_result.scores.iloc[:, :n_pcs]], axis=1)


def plot_pca_variance(pca_result: ParcelPCA, thresholds: Sequence[float] = (0.96, 0.99)) -> tuple[Figure, np.ndarray]:
    """Plot individual and cumulative explained variance."""
    for threshold in thresholds:
        if not 0.0 < threshold <= 1.0:
            raise ValueError(f"All thresholds must be in (0, 1], got {threshold}")

    variance = pca_result.variance
    x = np.arange(1, len(variance) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    axes[0].bar(x, 100 * variance["explained_variance_ratio"].to_numpy())
    axes[0].set(xlabel="Principal component", ylabel="Explained variance (%)", title="Scree plot")
    axes[0].set_xlim(0.5, len(x) + 0.5)

    cumulative = 100 * variance["cumulative_variance_ratio"].to_numpy()
    axes[1].plot(x, cumulative, marker="o", markersize=3)
    for threshold in thresholds:
        n_components = components_for_variance(pca_result, threshold)
        axes[1].axhline(100 * threshold, linestyle="--", linewidth=1)
        axes[1].axvline(n_components, linestyle=":", linewidth=1)
        axes[1].annotate(f"{100 * threshold:.0f}%: {n_components} PCs", (n_components, 100 * threshold), xytext=(5, 5), textcoords="offset points")
    axes[1].set(xlabel="Number of components", ylabel="Cumulative variance (%)", title="Cumulative explained variance")
    axes[1].set_xlim(0.5, len(x) + 0.5)
    axes[1].set_ylim(0, 101)
    return fig, axes


def plot_predictor_outcome(
    predictor: pd.Series,
    outcomes: pd.DataFrame,
    *,
    subject_col: str = "subj_id",
    outcome_col: str = "cgi_change",
    ax: Axes | None = None,
) -> tuple[Figure, Axes]:
    """Plot one predictor against outcome with a least-squares line and Pearson r."""
    predictor_name = predictor.name or "predictor"
    x = pd.to_numeric(predictor, errors="raise").astype(float)
    y = _align_outcomes(x.index, outcomes, subject_col, outcome_col)
    correlation = correlate_predictors(x.to_frame(predictor_name), outcomes, subject_col=subject_col, outcome_col=outcome_col).iloc[0]

    if ax is None:
        fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    else:
        fig = ax.figure
    ax.scatter(x, y)
    slope, intercept = np.polyfit(x.to_numpy(), y.to_numpy(), deg=1)
    line_x = np.linspace(x.min(), x.max(), 100)
    ax.plot(line_x, intercept + slope * line_x)
    ax.set(xlabel=predictor_name, ylabel=outcome_col, title=f"r = {correlation['r']:.2f}, p = {correlation['p_value']:.3g}")
    return fig, ax


def plot_global_outcome(
    summary: ParcelSummary,
    outcomes: pd.DataFrame,
    *,
    subject_col: str = "subj_id",
    outcome_col: str = "cgi_change",
) -> tuple[Figure, np.ndarray]:
    """Plot both global E summaries against the clinical outcome."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for ax, predictor_name in zip(axes, summary.global_metrics.columns, strict=True):
        plot_predictor_outcome(summary.global_metrics[predictor_name], outcomes, subject_col=subject_col, outcome_col=outcome_col, ax=ax)
    return fig, axes


def load_hdf5_mesh(
    hdf5_file: str | Path,
    *,
    mesh_key: str = "mesh_roi",
    mesh_loader: Callable | None = None,
):
    """Load a SimNIBS Msh stored under ``mesh_key`` in a gPC HDF5 file."""
    path = Path(hdf5_file)
    if not path.is_file() and mesh_loader is None:
        raise FileNotFoundError(f"HDF5 file does not exist: {path}")
    if mesh_loader is None:
        import simnibs

        mesh_loader = simnibs.Msh.read_hdf5
    try:
        return mesh_loader(str(hdf5_file), mesh_key)
    except (KeyError, OSError, ValueError) as exc:
        raise type(exc)(f"Could not load HDF5 mesh {mesh_key!r} from {hdf5_file}: {exc}") from exc


def _get_scalar_field(mesh, field_name: str, subject_id: str, *, require_nonnegative: bool) -> tuple[np.ndarray, str]:
    if field_name not in mesh.field:
        raise KeyError(f"Field {field_name!r} not found for {subject_id}; available: {sorted(mesh.field.keys())}")
    element_field_names = {field.field_name for field in mesh.elmdata}
    node_field_names = {field.field_name for field in mesh.nodedata}
    if field_name in element_field_names and field_name in node_field_names:
        raise ValueError(f"Field {field_name!r} exists as both element and node data for {subject_id}")
    if field_name in element_field_names:
        location = "element"
        expected_size = mesh.elm.nr
    elif field_name in node_field_names:
        location = "node"
        expected_size = mesh.nodes.nr
    else:
        raise ValueError(f"Could not determine the location of field {field_name!r} for {subject_id}")

    values = np.asarray(mesh.field[field_name].value, dtype=float).squeeze()
    if values.ndim != 1:
        raise ValueError(f"Field {field_name!r} must be scalar for {subject_id}, got shape {values.shape}")
    if values.size != expected_size:
        raise ValueError(f"Field {field_name!r} has {values.size} values but {location} count is {expected_size} for {subject_id}")
    if not np.isfinite(values).all():
        raise ValueError(f"Field {field_name!r} contains non-finite values for {subject_id}")
    if require_nonnegative and (values < 0).any():
        raise ValueError(f"Field {field_name!r} contains negative values for {subject_id}")
    return values, location


def _get_spatial_weights(mesh, location: str, subject_id: str) -> np.ndarray:
    if location == "element":
        element_types = np.unique(np.asarray(mesh.elm.elm_type)).astype(int)
        if element_types.size != 1:
            raise ValueError(
                f"Element field for {subject_id} spans mixed element types {element_types.tolist()}; "
                "their size weights have different physical units. Crop to one element type first."
            )
        weights_obj = mesh.elements_volumes_and_areas()
        expected_size = mesh.elm.nr
    elif location == "node":
        weights_obj = mesh.nodes_areas()
        expected_size = mesh.nodes.nr
    else:
        raise ValueError(f"Unknown field location {location!r} for {subject_id}")

    weights = np.asarray(getattr(weights_obj, "value", weights_obj), dtype=float).squeeze()
    if weights.ndim != 1 or weights.size != expected_size:
        raise ValueError(f"Invalid spatial-weight shape for {subject_id}: {weights.shape}; expected ({expected_size},)")
    if not np.isfinite(weights).all() or (weights <= 0).any():
        raise ValueError(f"Spatial weights must be finite and positive for {subject_id}")
    return weights


def _validate_parcel_masks(
    parcel_masks: Mapping[str, np.ndarray],
    n_values: int,
    subject_id: str,
) -> dict[str, np.ndarray]:
    if not parcel_masks:
        raise ValueError(f"No parcel masks were returned for {subject_id}")
    masks = {}
    coverage = np.zeros(n_values, dtype=np.int16)
    for parcel_name in sorted(parcel_masks):
        mask = np.asarray(parcel_masks[parcel_name], dtype=bool).squeeze()
        if mask.ndim != 1 or mask.size != n_values:
            raise ValueError(f"Parcel {parcel_name!r} has shape {mask.shape}; expected ({n_values},) for {subject_id}")
        if not mask.any():
            raise ValueError(f"Parcel {parcel_name!r} is empty for {subject_id}")
        masks[parcel_name] = mask
        coverage += mask
    if (coverage > 1).any():
        raise ValueError(f"Parcel masks overlap at {(coverage > 1).sum()} field locations for {subject_id}")
    return masks


def _parcel_masks_from_labels(
    labels: np.ndarray,
    *,
    label_names: Mapping[int, str] | None,
    exclude_labels: Sequence[int],
) -> dict[str, np.ndarray]:
    labels = np.asarray(labels).squeeze()
    if labels.ndim != 1:
        raise ValueError(f"Parcel labels must be one-dimensional, got shape {labels.shape}")
    if not np.issubdtype(labels.dtype, np.number) or not np.isfinite(labels).all():
        raise ValueError("Parcel labels must be finite numeric values")
    if not np.allclose(labels, np.rint(labels)):
        raise ValueError("Parcel labels must be integer-valued")
    labels = np.rint(labels).astype(int)
    selected_labels = sorted(set(np.unique(labels)).difference(int(label) for label in exclude_labels))
    if len(selected_labels) < 2:
        raise ValueError(f"At least two parcel labels are required, found {selected_labels}")

    if label_names is not None:
        missing_names = sorted(set(selected_labels).difference(label_names))
        extra_names = sorted(set(label_names).difference(selected_labels))
        if missing_names or extra_names:
            raise ValueError(f"Parcel-name mismatch; missing names={missing_names}, extra names={extra_names}")
        names = {label: str(label_names[label]) for label in selected_labels}
    else:
        names = {label: f"parcel_{label}" for label in selected_labels}
    if len(set(names.values())) != len(names):
        raise ValueError("Parcel names must be unique")
    return {names[label]: labels == label for label in selected_labels}


def _summarize_one_subject(
    field: np.ndarray,
    spatial_weights: np.ndarray,
    masks: Mapping[str, np.ndarray],
    percentile: float,
    subject_id: str,
) -> tuple[pd.Series, pd.Series]:
    p95 = {}
    sizes = {}
    for parcel_name, mask in masks.items():
        parcel_values = field[mask]
        if parcel_values.size == 0:
            raise ValueError(f"Parcel {parcel_name!r} is empty for {subject_id}")
        p95[parcel_name] = float(np.percentile(parcel_values, percentile))
        sizes[parcel_name] = float(spatial_weights[mask].sum())
    return pd.Series(p95, dtype=float), pd.Series(sizes, dtype=float)


def _validate_subject_parcel_frame(parcel_p95: pd.DataFrame) -> None:
    if parcel_p95.empty:
        raise ValueError("parcel_p95 is empty")
    if parcel_p95.shape[0] < 3:
        raise ValueError(f"PCA requires at least 3 subjects, got {parcel_p95.shape[0]}")
    if parcel_p95.shape[1] < 2:
        raise ValueError(f"PCA requires at least 2 parcels, got {parcel_p95.shape[1]}")
    if parcel_p95.index.has_duplicates:
        raise ValueError(f"Duplicate subject IDs: {parcel_p95.index[parcel_p95.index.duplicated()].unique().tolist()}")
    if parcel_p95.columns.has_duplicates:
        raise ValueError(f"Duplicate parcels: {parcel_p95.columns[parcel_p95.columns.duplicated()].unique().tolist()}")
    _require_finite_frame(parcel_p95, "parcel P95 matrix")
    if (parcel_p95 < 0).any().any():
        raise ValueError("parcel_p95 contains negative values")


def _require_finite_frame(frame: pd.DataFrame, name: str) -> None:
    try:
        values = frame.to_numpy(dtype=float)
    except (TypeError, ValueError) as exc:
        raise TypeError(f"{name} must be numeric") from exc
    if not np.isfinite(values).all():
        bad_rows, bad_cols = np.where(~np.isfinite(values))
        examples = [(str(frame.index[i]), str(frame.columns[j])) for i, j in zip(bad_rows[:5], bad_cols[:5], strict=True)]
        raise ValueError(f"{name} contains non-finite values; examples: {examples}")


def _align_outcomes(subject_index: pd.Index, outcomes: pd.DataFrame, subject_col: str, outcome_col: str) -> pd.Series:
    required = {subject_col, outcome_col}
    missing_columns = sorted(required.difference(outcomes.columns))
    if missing_columns:
        raise KeyError(f"outcomes is missing columns: {missing_columns}")
    if outcomes[subject_col].isna().any():
        raise ValueError(f"{subject_col!r} contains missing IDs")
    duplicate_ids = outcomes.loc[outcomes[subject_col].duplicated(keep=False), subject_col].astype(str).unique().tolist()
    if duplicate_ids:
        raise ValueError(f"outcomes contains duplicate subject IDs: {duplicate_ids}")

    outcome = outcomes.set_index(subject_col)[outcome_col]
    outcome.index = outcome.index.astype(str)
    subject_index = pd.Index(subject_index.astype(str))
    missing_outcomes = subject_index.difference(outcome.index).tolist()
    extra_outcomes = outcome.index.difference(subject_index).tolist()
    if missing_outcomes or extra_outcomes:
        raise ValueError(f"Subject mismatch; missing outcomes={missing_outcomes}, extra outcomes={extra_outcomes}")
    outcome = pd.to_numeric(outcome.loc[subject_index], errors="raise").astype(float)
    if not np.isfinite(outcome.to_numpy()).all():
        raise ValueError(f"{outcome_col!r} contains non-finite values")
    if np.ptp(outcome.to_numpy()) == 0:
        raise ValueError(f"{outcome_col!r} has zero variance")
    return outcome


def _pearson_fisher_ci(r: float, n: int, confidence: float = 0.95) -> tuple[float, float]:
    if n <= 3:
        return float("nan"), float("nan")
    clipped_r = np.clip(r, -1 + np.finfo(float).eps, 1 - np.finfo(float).eps)
    z = np.arctanh(clipped_r)
    critical = norm.ppf(0.5 + confidence / 2)
    half_width = critical / np.sqrt(n - 3)
    return float(np.tanh(z - half_width)), float(np.tanh(z + half_width))


def _benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    values = p_values.to_numpy(dtype=float)
    if not ((0 <= values) & (values <= 1)).all():
        raise ValueError("p-values must be between 0 and 1")
    order = np.argsort(values)
    ranked = values[order]
    adjusted_ranked = np.minimum.accumulate((ranked * len(values) / np.arange(1, len(values) + 1))[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return pd.Series(adjusted, index=p_values.index, name="p_fdr_bh")


In [ ]:
def load_cogscores_batch(fn: str) -> pd.DataFrame:
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)

    cleaned_columns = df.columns.str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = ["subjid", "cgi_start", "cgi_end", "date_start", "date_end (acute)", "mrn"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    for column in ("cgi_start", "cgi_end"):
        if df[column].isna().any():
            idx_bad_rows = df.index[df[column].isna()].tolist()
            print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{column}': {idx_bad_rows}")
            df = df.drop(index=idx_bad_rows).copy()

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")
    df = df.drop(columns=["mrn"])

    return df



In [ ]:
import pandas as pd
from pathlib import Path

def get_hdf5_paths(df: pd.DataFrame, column: str = "full_file_path") -> list[str]:
    if column not in df.columns:
        raise KeyError(f"DataFrame does not contain column {column!r}")

    return df.loc[df[column].str.endswith(".hdf5", na=False), column].tolist()

def load_cogscores_batch(fn: str) -> pd.DataFrame:
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)

    cleaned_columns = df.columns.str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = ["subjid", "cgi_start", "cgi_end", "date_start", "date_end (acute)", "mrn"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    for column in ("cgi_start", "cgi_end"):
        if df[column].isna().any():
            idx_bad_rows = df.index[df[column].isna()].tolist()
            print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{column}': {idx_bad_rows}")
            df = df.drop(index=idx_bad_rows).copy()

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")
    df = df.drop(columns=["mrn"])

    return df

def add_base_subjid(df: pd.DataFrame, source_col: str = "subjid") -> pd.DataFrame:
    
    df = df.copy()

    if source_col not in df.columns:
        raise KeyError(f"Required subject-ID column '{source_col}' is missing.")

    df[source_col] = df[source_col].astype("string").str.strip()

    if df[source_col].isna().any():
        bad_rows = df.index[df[source_col].isna()].tolist()
        raise ValueError(f"Missing subject IDs in column '{source_col}' at rows: {bad_rows}")

    if df[source_col].eq("").any():
        bad_rows = df.index[df[source_col].eq("")].tolist()
        raise ValueError(f"Empty subject IDs in column '{source_col}' at rows: {bad_rows}")

    df["subjid_base"] = df[source_col].str.replace(r"-\d{3}$", "", regex=True)

    return df

# CGI scores
df_cog = load_cogscores_batch(
    "/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx"
)

# identify the paths
query = pd.read_csv('/projects/p32903/Alex2/tests/sql-table/query_result.csv')
query           = query[query.dataset_root == 'STU00225089']
query           = query[query.model_type_dir == 'skin_single']
query_static    = query[query.step == 'step_0']
query_adaptive  = query[query.step == 'step_2']

# collect the paths 
hdf5_paths_static   = get_hdf5_paths(query_static)
hdf5_paths_adaptive = get_hdf5_paths(query_adaptive)


from pathlib import Path
from example_usage import run_all_atlases
import os 

# Existing objects:
# hdf5_paths_adaptive: list of tdcs_uq_gpc.hdf5 paths
# df_cog: DataFrame containing subjid and either:
#         - cgi_change, or
#         - cgi_start and cgi_end

SUBJECTS_DIR = Path("/projects/p32903/Alex2/datasets/STU00225089/subjectsdir")
OUTPUT_DIR = Path("/projects/p32903/Alex2/ect-workflow/05_analysis/output")

# Usually this can remain empty.
# An explicit entry is only required if multiple FreeSurfer directories share
# the same base ID and the correct directory would otherwise be ambiguous.
SUBJECT_MAP = {
    # "subj-cat-001-002": "subj-cat-001-001",
}

results = run_all_atlases(
    hdf5_files=hdf5_paths_static,
    cognitive=df_cog,
    subjects_dir=SUBJECTS_DIR,
    output_dir=OUTPUT_DIR,
    subject_map=SUBJECT_MAP,
    include_surface_hcp=True,
    run_unweighted_sensitivity=True,
    change_direction="end_minus_start",
    freesurfer_lut='/projects/p32903/Alex2/ecstim/resources/FreeSurferColorLUT.txt'
)

# Primary volumetric DK analysis using tetrahedron-volume-weighted P95
dk = results["volume_aparc_weighted"]

print("PC counts:")
print(dk.components_by_threshold)

print("\nGlobal E correlations:")
print(
    dk.correlations.loc[
        ["global_mean_E_unweighted", "global_mean_E_weighted"]
    ]
)

print("\nAll global and PC correlations:")
print(dk.correlations)

print("\nPCA explained variance:")
print(dk.pca.variance)

print("\nPC loadings:")
print(dk.pca.loadings)

In [ ]:
from pathlib import Path
from typing import Literal

import pandas as pd

def add_base_subjid(df: pd.DataFrame, source_col: str = "subjid") -> pd.DataFrame:
    
    df = df.copy()

    if source_col not in df.columns:
        raise KeyError(f"Required subject-ID column '{source_col}' is missing.")

    df[source_col] = df[source_col].astype("string").str.strip()

    if df[source_col].isna().any():
        bad_rows = df.index[df[source_col].isna()].tolist()
        raise ValueError(f"Missing subject IDs in column '{source_col}' at rows: {bad_rows}")

    if df[source_col].eq("").any():
        bad_rows = df.index[df[source_col].eq("")].tolist()
        raise ValueError(f"Empty subject IDs in column '{source_col}' at rows: {bad_rows}")

    df["subjid_base"] = df[source_col].str.replace(r"-\d{3}$", "", regex=True)

    return df


def load_cogscores_batch(
    fn: str | Path,
    *,
    change_direction: Literal["end_minus_start", "start_minus_end"] = "end_minus_start",) -> pd.DataFrame:
    """Load complete CGI treatment courses and calculate CGI change."""
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)
    cleaned_columns = df.columns.astype(str).str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = [
        "subjid",
        "cgi_start",
        "cgi_end",
        "date_start",
        "date_end (acute)",
        "mrn",
    ]
    missing_cols = sorted(set(required_cols).difference(df.columns))

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    if change_direction not in {"end_minus_start", "start_minus_end"}:
        raise ValueError(
            "change_direction must be 'end_minus_start' or 'start_minus_end'"
        )

    df["clinical_row_index"] = df.index
    df[["cgi_start", "cgi_end"]] = df[["cgi_start", "cgi_end"]].replace(
        r"^\s*$",
        pd.NA,
        regex=True,
    )

    complete_mask = df[["cgi_start", "cgi_end"]].notna().all(axis=1)
    excluded = df.loc[
        ~complete_mask,
        ["clinical_row_index", "subjid", "cgi_start", "cgi_end"],
    ]
    df = df.loc[complete_mask].copy()

    print(f"Clinical rows loaded: {len(complete_mask)}")
    print(f"Complete treatment courses retained: {len(df)}")
    print(f"Incomplete treatment courses excluded: {len(excluded)}")

    if not excluded.empty:
        print("\nExcluded clinical rows:")
        print(excluded.to_string(index=False))

    if df.empty:
        raise ValueError("No treatment courses have both CGI start and CGI end")

    df[["cgi_start", "cgi_end"]] = df[
        ["cgi_start", "cgi_end"]
    ].apply(pd.to_numeric, errors="raise")

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")

    if change_direction == "end_minus_start":
        df["cgi_change"] = df["cgi_end"] - df["cgi_start"]
    else:
        df["cgi_change"] = df["cgi_start"] - df["cgi_end"]

    return df.drop(columns=["mrn"]).reset_index(drop=True)


def find_cgi_course_conflicts(df_cog: pd.DataFrame) -> pd.DataFrame:
    """Return complete treatment courses for people with differing CGI changes."""
    required = {
        "subjid",
        "subjid_base",
        "cgi_start",
        "cgi_end",
        "cgi_change",
    }
    missing = sorted(required.difference(df_cog.columns))

    if missing:
        raise KeyError(f"df_cog is missing required columns: {missing}")

    outcome_counts = (
        df_cog.groupby("subjid_base")["cgi_change"]
        .nunique(dropna=False)
    )
    conflicting_subjects = outcome_counts[outcome_counts > 1].index

    columns = [
        "clinical_row_index",
        "subjid",
        "subjid_base",
        "date_start",
        "date_end (acute)",
        "cgi_start",
        "cgi_end",
        "cgi_change",
    ]

    return (
        df_cog.loc[df_cog["subjid_base"].isin(conflicting_subjects), columns]
        .sort_values(["subjid_base", "date_start", "clinical_row_index"])
        .reset_index(drop=True)
    )


df_cog = load_cogscores_batch(
    "/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx"
)

cgi_conflicts = find_cgi_course_conflicts(df_cog)

print("\nComplete treatment courses with conflicting person-level outcomes:")
print(cgi_conflicts.to_string(index=False))

In [ ]:
from typing import Literal
import pandas as pd
from simnibs_parcel_analysis import add_subject_id_columns
from pathlib import Path
from pandas import pd 

def load_cogscores_batch(fn: str) -> pd.DataFrame:
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)

    cleaned_columns = df.columns.str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = ["subjid", "cgi_start", "cgi_end", "date_start", "date_end (acute)", "mrn"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    for column in ("cgi_start", "cgi_end"):
        if df[column].isna().any():
            idx_bad_rows = df.index[df[column].isna()].tolist()
            print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{column}': {idx_bad_rows}")
            df = df.drop(index=idx_bad_rows).copy()

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")
    df = df.drop(columns=["mrn"])

    return df

def add_base_subjid(df: pd.DataFrame, source_col: str = "subjid") -> pd.DataFrame:
    
    df = df.copy()

    if source_col not in df.columns:
        raise KeyError(f"Required subject-ID column '{source_col}' is missing.")

    df[source_col] = df[source_col].astype("string").str.strip()

    if df[source_col].isna().any():
        bad_rows = df.index[df[source_col].isna()].tolist()
        raise ValueError(f"Missing subject IDs in column '{source_col}' at rows: {bad_rows}")

    if df[source_col].eq("").any():
        bad_rows = df.index[df[source_col].eq("")].tolist()
        raise ValueError(f"Empty subject IDs in column '{source_col}' at rows: {bad_rows}")

    df["subjid_base"] = df[source_col].str.replace(r"-\d{3}$", "", regex=True)

    return df

# CGI scores
df_cog = load_cogscores_batch(
    "/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx"
)

def prepare_complete_cgi_courses(
    clinical: pd.DataFrame,
    *,
    id_col: str = "subjid",
    start_col: str = "cgi_start",
    end_col: str = "cgi_end",
    outcome_col: str = "cgi_change",
    direction: Literal["end_minus_start", "start_minus_end"] = "end_minus_start") -> pd.DataFrame:
    """Keep treatment courses with both CGI values and calculate CGI change."""
    required = {id_col, start_col, end_col}
    missing = sorted(required.difference(clinical.columns))

    if missing:
        raise KeyError(f"Clinical data are missing required columns: {missing}")

    if direction not in {"end_minus_start", "start_minus_end"}:
        raise ValueError("direction must be 'end_minus_start' or 'start_minus_end'")

    result = clinical.copy()
    result["clinical_row_index"] = result.index

    # Treat whitespace-only cells as missing.
    result[[start_col, end_col]] = result[[start_col, end_col]].replace(r"^\s*$", pd.NA, regex=True)

    complete_mask = result[[start_col, end_col]].notna().all(axis=1)
    excluded = result.loc[~complete_mask, ["clinical_row_index", id_col, start_col, end_col]]
    result = result.loc[complete_mask].copy()

    if result.empty:
        raise ValueError("No treatment courses have both CGI start and CGI end values")

    result[[start_col, end_col]] = result[[start_col, end_col]].apply(pd.to_numeric, errors="raise")
    result = add_subject_id_columns(result, source_col=id_col, require_scan=False)

    if direction == "end_minus_start":
        result[outcome_col] = result[end_col] - result[start_col]
    else:
        result[outcome_col] = result[start_col] - result[end_col]

    print(f"Clinical rows before filtering: {len(clinical)}")
    print(f"Complete treatment courses retained: {len(result)}")
    print(f"Incomplete treatment courses excluded: {len(excluded)}")

    if not excluded.empty:
        print("\nExcluded rows:")
        print(excluded.to_string(index=False))

    return result.reset_index(drop=True)


# Test the function by itself.
df_cog_complete = prepare_complete_cgi_courses(
    df_cog,
    direction="end_minus_start",
)

# Inspect subjects with multiple complete treatment courses.
course_summary = (
    df_cog_complete
    .groupby("subjid_base", as_index=False)
    .agg(
        n_complete_courses=("cgi_change", "size"),
        n_unique_cgi_changes=("cgi_change", "nunique"),
        cgi_changes=("cgi_change", list),
    )
)

multiple_courses = course_summary.loc[course_summary["n_complete_courses"] > 1]

print("\nSubjects with multiple complete treatment courses:")
print(multiple_courses.to_string(index=False))

print("\nCleaned treatment-course rows:")
print(
    df_cog_complete[
        ["clinical_row_index", "subjid", "subjid_base", "cgi_start", "cgi_end", "cgi_change"]
    ].to_string(index=False)
)